<a href="https://colab.research.google.com/github/kawastony/Quadratic-Mechanism-Lens/blob/main/TIFA_Paper_B_Calc_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# ============================================================
# TIFA / cosine-quintessence + DESI DR1 BAO likelihood
# One self-contained script
#
# Conventions:
#   Mpl = 1
#   H0 = 1 today
#   rho_c0 = 3
#   rho_m(a) = 3 Omega_m a^-3
#   rho_r(a) = 3 Omega_r a^-4
#
# Friedmann:
#   H^2 = (rho_m + rho_r + V) / (3 - 0.5 phip^2)
#
# Scalar:
#   V(phi) = L4 * [1 - cos(phi/fE)]
#
# This script:
#   1) Solves the KG+Friedmann background
#   2) Shoots L4 so H(a=1)=1
#   3) Computes H(z), q(z), w_phi(z)
#   4) Builds DESI BAO observables
#   5) Evaluates chi^2 using the 12-point covariance
#   6) Scans phi_ini if desired
# ============================================================

import numpy as np
from scipy.integrate import solve_ivp, quad
from scipy.optimize import brentq

# ============================================================
# SECTION A — COSMOLOGICAL PARAMETERS
# ============================================================

Omega_m = 0.315
Omega_r = 9.0e-5
Omega_phi0_target = 1.0 - Omega_m - Omega_r
Omega_L = 1.0 - Omega_m - Omega_r

fE = 2.0           # decay constant in reduced Planck units
z_ini_default = 1000.0

# Sound horizon:
# IMPORTANT:
# DESI BAO observables are D/rd or H*rd in physical analyses.
# In your previous setup the data vector is already in D/rd and DH/rd,
# so here we compute dimensionless D_M/rd etc. by introducing rd_dimless.
#
# Since H0=1 units are abstract, rd_dimless acts as the calibration parameter.
# If you want Planck-anchored rd, set this accordingly.
# If you want BAO-only with free H0*rd, this becomes a nuisance parameter.
#
# For now we keep rd_dimless explicit.
rd_dimless_default = 0.0333   # placeholder dimensionless rd * H0/c-type scale

c_light = 1.0  # in these dimensionless units we absorb c consistently into rd choice


# ============================================================
# SECTION B — SCALAR POTENTIAL
# ============================================================

def V(phi, L4):
    return L4 * (1.0 - np.cos(phi / fE))

def dV_dphi(phi, L4):
    return (L4 / fE) * np.sin(phi / fE)

def d2V_dphi2(phi, L4):
    return (L4 / fE**2) * np.cos(phi / fE)


# ============================================================
# SECTION C — BACKGROUND DENSITIES
# ============================================================

def rho_m_of_a(a):
    return 3.0 * Omega_m / a**3

def rho_r_of_a(a):
    return 3.0 * Omega_r / a**4


# ============================================================
# SECTION D — EXACT FRIEDMANN / FLUID QUANTITIES
# ============================================================

def H_sq(a, phi, phip, L4):
    denom = 3.0 - 0.5 * phip**2
    if np.any(denom <= 0):
        return np.nan
    return (rho_m_of_a(a) + rho_r_of_a(a) + V(phi, L4)) / denom

def scalar_energy_pressure(a, phi, phip, L4):
    H2 = H_sq(a, phi, phip, L4)
    K = 0.5 * H2 * phip**2
    VV = V(phi, L4)
    rho_phi = K + VV
    p_phi = K - VV
    return rho_phi, p_phi, K, VV, H2

def w_phi(a, phi, phip, L4):
    rho_phi, p_phi, _, _, _ = scalar_energy_pressure(a, phi, phip, L4)
    return p_phi / rho_phi

def w_total(a, phi, phip, L4):
    rho_phi, p_phi, _, _, _ = scalar_energy_pressure(a, phi, phip, L4)
    rho_tot = rho_m_of_a(a) + rho_r_of_a(a) + rho_phi
    p_tot = rho_r_of_a(a) / 3.0 + p_phi
    return p_tot / rho_tot

def q_of_state(a, phi, phip, L4):
    return 0.5 * (1.0 + 3.0 * w_total(a, phi, phip, L4))

def Hprime_over_H(a, phi, phip, L4):
    wt = w_total(a, phi, phip, L4)
    return -1.5 * (1.0 + wt)


# ============================================================
# SECTION E — KG EQUATION IN N = ln a
# ============================================================

def rhs_background(N, y, L4):
    phi, phip = y
    a = np.exp(N)

    H2 = H_sq(a, phi, phip, L4)
    if not np.isfinite(H2) or H2 <= 0:
        return [np.nan, np.nan]

    HpH = Hprime_over_H(a, phi, phip, L4)
    phipp = -(3.0 + HpH) * phip - dV_dphi(phi, L4) / H2
    return [phip, phipp]


# ============================================================
# SECTION F — BACKGROUND SOLVER
# ============================================================

def integrate_background(phi_ini,
                         L4,
                         z_ini=z_ini_default,
                         phip_ini=0.0,
                         rtol=1e-9,
                         atol=1e-11,
                         dense_output=True,
                         method='RK45'):
    N_ini = np.log(1.0 / (1.0 + z_ini))
    N_end = 0.0

    sol = solve_ivp(
        fun=lambda N, y: rhs_background(N, y, L4),
        t_span=(N_ini, N_end),
        y0=[phi_ini, phip_ini],
        dense_output=dense_output,
        rtol=rtol,
        atol=atol,
        method=method
    )
    return sol

def today_diagnostics(sol, L4):
    if (not sol.success) or (sol.sol is None):
        return None

    phi0, phip0 = sol.sol(0.0)
    a0 = 1.0

    rho_phi, p_phi, K0, V0, H20 = scalar_energy_pressure(a0, phi0, phip0, L4)
    rho_m0 = rho_m_of_a(a0)
    rho_r0 = rho_r_of_a(a0)
    rho_tot0 = rho_m0 + rho_r0 + rho_phi
    p_tot0 = rho_r0 / 3.0 + p_phi

    return {
        "phi0": float(phi0),
        "phip0": float(phip0),
        "H0": float(np.sqrt(H20)),
        "H0_sq": float(H20),
        "rho_m0": float(rho_m0),
        "rho_r0": float(rho_r0),
        "rho_phi0": float(rho_phi),
        "p_phi0": float(p_phi),
        "rho_tot0": float(rho_tot0),
        "K0": float(K0),
        "V0": float(V0),
        "wphi0": float(p_phi / rho_phi),
        "wtot0": float(p_tot0 / rho_tot0),
        "q0": float(0.5 * (1.0 + 3.0 * p_tot0 / rho_tot0)),
        "Omega_m0_eff": float(rho_m0 / rho_tot0),
        "Omega_r0_eff": float(rho_r0 / rho_tot0),
        "Omega_phi0_eff": float(rho_phi / rho_tot0),
        "closure_error": float(rho_tot0 - 3.0 * H20)
    }

def H0_minus_1_for_root(L4, phi_ini, z_ini=z_ini_default, phip_ini=0.0):
    sol = integrate_background(phi_ini, L4, z_ini=z_ini, phip_ini=phip_ini)
    if (not sol.success) or (sol.sol is None):
        return np.nan
    diag = today_diagnostics(sol, L4)
    if diag is None or not np.isfinite(diag["H0"]):
        return np.nan
    return diag["H0"] - 1.0

def shoot_L4_for_H0_eq_1(phi_ini,
                         L4_min=1e-8,
                         L4_max=1e3,
                         z_ini=z_ini_default,
                         phip_ini=0.0):
    def f(L4):
        return H0_minus_1_for_root(L4, phi_ini, z_ini=z_ini, phip_ini=phip_ini)

    a = L4_min
    b = L4_max
    fa = f(a)
    fb = f(b)

    for _ in range(40):
        if np.isfinite(fa) and np.isfinite(fb) and fa * fb < 0:
            break
        a /= 10.0
        b *= 10.0
        fa = f(a)
        fb = f(b)
    else:
        raise RuntimeError("Could not bracket root for L4.")

    L4_star = brentq(f, a, b, xtol=1e-12, rtol=1e-10, maxiter=200)
    sol_star = integrate_background(phi_ini, L4_star, z_ini=z_ini, phip_ini=phip_ini)

    if not sol_star.success:
        raise RuntimeError("Integration failed after L4 shooting.")

    return L4_star, sol_star, today_diagnostics(sol_star, L4_star)


# ============================================================
# SECTION G — REDSHIFT-SPACE DIAGNOSTICS
# ============================================================

def state_at_z(z, sol, L4):
    a = 1.0 / (1.0 + z)
    N = np.log(a)
    phi, phip = sol.sol(N)

    rho_phi, p_phi, K, VV, H2 = scalar_energy_pressure(a, phi, phip, L4)
    rho_m = rho_m_of_a(a)
    rho_r = rho_r_of_a(a)
    rho_tot = rho_m + rho_r + rho_phi
    p_tot = rho_r / 3.0 + p_phi

    return {
        "z": float(z),
        "a": float(a),
        "phi": float(phi),
        "phip": float(phip),
        "H": float(np.sqrt(H2)),
        "H_sq": float(H2),
        "rho_m": float(rho_m),
        "rho_r": float(rho_r),
        "rho_phi": float(rho_phi),
        "p_phi": float(p_phi),
        "rho_tot": float(rho_tot),
        "p_tot": float(p_tot),
        "K": float(K),
        "V": float(VV),
        "w_phi": float(p_phi / rho_phi),
        "w_tot": float(p_tot / rho_tot),
        "q": float(0.5 * (1.0 + 3.0 * p_tot / rho_tot)),
        "Omega_m": float(rho_m / rho_tot),
        "Omega_r": float(rho_r / rho_tot),
        "Omega_phi": float(rho_phi / rho_tot),
        "closure_error": float(rho_tot - 3.0 * H2),
    }

def H_of_z(z, sol, L4):
    return state_at_z(z, sol, L4)["H"]

def q_of_z(z, sol, L4):
    return state_at_z(z, sol, L4)["q"]

def wphi_of_z(z, sol, L4):
    return state_at_z(z, sol, L4)["w_phi"]


# ============================================================
# SECTION H — LCDM REFERENCE
# ============================================================

def H_sq_lcdm(a):
    rho_m = rho_m_of_a(a)
    rho_r = rho_r_of_a(a)
    rho_L = 3.0 * Omega_L
    return (rho_m + rho_r + rho_L) / 3.0

def H_lcdm_of_z(z):
    a = 1.0 / (1.0 + z)
    return np.sqrt(H_sq_lcdm(a))

def state_lcdm_at_z(z):
    a = 1.0 / (1.0 + z)
    H2 = H_sq_lcdm(a)

    rho_m = rho_m_of_a(a)
    rho_r = rho_r_of_a(a)
    rho_L = 3.0 * Omega_L
    rho_tot = rho_m + rho_r + rho_L
    p_tot = rho_r / 3.0 - rho_L

    return {
        "z": float(z),
        "a": float(a),
        "H": float(np.sqrt(H2)),
        "q": float(0.5 * (1.0 + 3.0 * p_tot / rho_tot)),
        "w_tot": float(p_tot / rho_tot)
    }

def delta_q(z_array, sol, L4):
    z_array = np.asarray(z_array)
    out = np.empty_like(z_array, dtype=float)
    for i, z in enumerate(z_array):
        out[i] = q_of_z(z, sol, L4) - state_lcdm_at_z(z)["q"]
    return out

def delta_H_over_H_lcdm(z_array, sol, L4):
    z_array = np.asarray(z_array)
    out = np.empty_like(z_array, dtype=float)
    for i, z in enumerate(z_array):
        Ht = H_of_z(z, sol, L4)
        Hl = H_lcdm_of_z(z)
        out[i] = (Ht - Hl) / Hl
    return out


# ============================================================
# SECTION I — DISTANCES FOR BAO
# ============================================================

def comoving_distance(z, sol, L4):
    # Flat universe: D_M = \int_0^z dz'/H(z')
    val, err = quad(lambda zp: 1.0 / H_of_z(zp, sol, L4), 0.0, z,
                    epsabs=1e-9, epsrel=1e-9, limit=300)
    return val

def DM_over_rd(z, sol, L4, rd_dimless=rd_dimless_default):
    return comoving_distance(z, sol, L4) / rd_dimless

def DH_over_rd(z, sol, L4, rd_dimless=rd_dimless_default):
    return (1.0 / H_of_z(z, sol, L4)) / rd_dimless

def DV_over_rd(z, sol, L4, rd_dimless=rd_dimless_default):
    DM = comoving_distance(z, sol, L4)
    DH = 1.0 / H_of_z(z, sol, L4)
    DV = (z * DM**2 * DH)**(1.0 / 3.0)
    return DV / rd_dimless


# ============================================================
# SECTION J — DESI DR1 BAO DATA VECTOR
# Order:
# 1.  BGS z=0.295: DV/rd
# 2-3. LRG1 z=0.510: DM/rd, DH/rd
# 4-5. LRG2 z=0.706: DM/rd, DH/rd
# 6-7. LRG3+ELG1 z=0.934: DM/rd, DH/rd
# 8-9. ELG2 z=1.317: DM/rd, DH/rd
# 10. QSO z=1.491: DV/rd
# 11-12. Lya z=2.330: DM/rd, DH/rd
# ============================================================

data_vector = np.array([
    7.93,
    13.62, 20.98,
    16.85, 20.08,
    21.71, 17.88,
    27.79, 13.82,
    26.07,
    39.71, 8.52
], dtype=float)

sigma = np.array([
    0.15,
    0.25, 0.61,
    0.32, 0.60,
    0.28, 0.35,
    0.69, 0.42,
    0.67,
    0.94, 0.17
], dtype=float)

C = np.array([
    [0.0225, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0.0625, -0.445*0.25*0.61, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, -0.445*0.25*0.61, 0.3721, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0.1024, -0.420*0.32*0.60, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, -0.420*0.32*0.60, 0.36, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0.0784, -0.389*0.28*0.35, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, -0.389*0.28*0.35, 0.1225, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0.4761, -0.444*0.69*0.42, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, -0.444*0.69*0.42, 0.1764, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0.4489, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.8836, -0.477*0.94*0.17],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, -0.477*0.94*0.17, 0.0289]
], dtype=float)

Cinv = np.linalg.inv(C)

def calculate_chi_squared(d, m, Cinv):
    diff = np.asarray(d) - np.asarray(m)
    return float(diff.T @ Cinv @ diff)


# ============================================================
# SECTION K — MODEL PREDICTION VECTOR IN DESI ORDER
# ============================================================

def bao_model_vector(sol, L4, rd_dimless=rd_dimless_default):
    return np.array([
        DV_over_rd(0.295, sol, L4, rd_dimless),

        DM_over_rd(0.510, sol, L4, rd_dimless),
        DH_over_rd(0.510, sol, L4, rd_dimless),

        DM_over_rd(0.706, sol, L4, rd_dimless),
        DH_over_rd(0.706, sol, L4, rd_dimless),

        DM_over_rd(0.934, sol, L4, rd_dimless),
        DH_over_rd(0.934, sol, L4, rd_dimless),

        DM_over_rd(1.317, sol, L4, rd_dimless),
        DH_over_rd(1.317, sol, L4, rd_dimless),

        DV_over_rd(1.491, sol, L4, rd_dimless),

        DM_over_rd(2.330, sol, L4, rd_dimless),
        DH_over_rd(2.330, sol, L4, rd_dimless),
    ], dtype=float)

def chi2_tifa(phi_ini, rd_dimless=rd_dimless_default, z_ini=z_ini_default):
    L4, sol, diag = shoot_L4_for_H0_eq_1(phi_ini, z_ini=z_ini)
    m = bao_model_vector(sol, L4, rd_dimless=rd_dimless)
    chi2 = calculate_chi_squared(data_vector, m, Cinv)
    return {
        "phi_ini": phi_ini,
        "phi_ini_over_pi": phi_ini / np.pi,
        "L4": L4,
        "sol": sol,
        "diag0": diag,
        "model_vector": m,
        "chi2": chi2
    }


# ============================================================
# SECTION L — OPTIONAL SCAN
# ============================================================

def scan_phi_ini(phi_over_pi_grid,
                 rd_dimless=rd_dimless_default,
                 z_ini=z_ini_default,
                 verbose=True):
    results = []

    for x in phi_over_pi_grid:
        phi_ini = x * np.pi
        try:
            out = chi2_tifa(phi_ini, rd_dimless=rd_dimless, z_ini=z_ini)
            results.append(out)
            if verbose:
                print(
                    f"phi_ini/pi={x:.4f} | "
                    f"chi2={out['chi2']:.6f} | "
                    f"L4={out['L4']:.8f} | "
                    f"phi0/pi={out['diag0']['phi0']/np.pi:.6f} | "
                    f"w0={out['diag0']['wphi0']:.6f} | "
                    f"q0={out['diag0']['q0']:.6f}"
                )
        except Exception as e:
            if verbose:
                print(f"phi_ini/pi={x:.4f} failed: {e}")

    return results

def best_result(results):
    if len(results) == 0:
        return None
    return min(results, key=lambda r: r["chi2"])


# ============================================================
# SECTION M — EXAMPLE RUN
# ============================================================

if __name__ == "__main__":
    # Example single run
    phi_ini_test = 0.34 * np.pi
    out = chi2_tifa(phi_ini_test, rd_dimless=rd_dimless_default)

    print("\n=== SINGLE RUN ===")
    print(f"phi_ini/pi          = {out['phi_ini_over_pi']:.8f}")
    print(f"L4                  = {out['L4']:.12g}")
    print(f"chi2                = {out['chi2']:.8f}")
    print(f"phi0/pi             = {out['diag0']['phi0']/np.pi:.8f}")
    print(f"wphi0               = {out['diag0']['wphi0']:.8f}")
    print(f"q0                  = {out['diag0']['q0']:.8f}")
    print(f"H0                  = {out['diag0']['H0']:.8f}")
    print(f"closure error       = {out['diag0']['closure_error']:.3e}")

    # Example scan
    grid = np.linspace(0.10, 0.90, 17)
    results = scan_phi_ini(grid, rd_dimless=rd_dimless_default, verbose=True)

    best = best_result(results)
    if best is not None:
        print("\n=== BEST SCAN RESULT ===")
        print(f"best phi_ini/pi     = {best['phi_ini_over_pi']:.8f}")
        print(f"best chi2           = {best['chi2']:.8f}")
        print(f"best L4             = {best['L4']:.12g}")
        print(f"best phi0/pi        = {best['diag0']['phi0']/np.pi:.8f}")
        print(f"best wphi0          = {best['diag0']['wphi0']:.8f}")
        print(f"best q0             = {best['diag0']['q0']:.8f}")